# Sélection du modèle et soumission

Nous avons choisi le modèle XGBoost qui obtient un score Kaggle de ...

In [ ]:
xgboost_model = XGBRegressor(**xgboost_best_params, objective='reg:squarederror', random_state=42, n_jobs=-1)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost', xgboost_model)
    ])

model_pipeline.fit(X_train, y)

y_pred = model_pipeline.predict(X_test)

kaggle_submission = pd.DataFrame()
kaggle_submission['Id'] = X_test.index
kaggle_submission['SalePrice'] = np.expm1(y_pred)
kaggle_submission.to_csv('Data/xgboost.csv', index=False)

# Déploiement

In [ ]:
# mlflow server --host 127.0.0.1 --port 8080

mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")
mlflow.set_experiment("House_Price_Prediction")

In [ ]:
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(xgboost_best_params)

    # Log the loss metric
    mlflow.log_metric("accuracy", xgboost_study.best_value)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "XGBoost")

    # Infer the model signature
    signature = infer_signature(X_train, model_pipeline.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=model_pipeline,
        signature=signature,
        input_example=X_train,
        registered_model_name="XGBoost_model",
        artifact_path="XGBoost_model"
    )